In [25]:
# Instala dependências necessárias (versão >= 0.2 do LangChain)
%pip install -q --upgrade \
    "langchain>=0.2" \
    langchain-text-splitters \
    langchain-openai \
    langchain-chroma \
    langchain-community \
    langchain-core \
    python-dotenv \
    pypdf \
    gitpython


Note: you may need to restart the kernel to use updated packages.


In [26]:
# Importações para RAG de código
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import os
from dotenv import load_dotenv
from git import Repo


In [27]:
# Carrega variáveis de ambiente (API key da OpenAI)
load_dotenv()
openai_key = os.getenv('OPENAI_API_KEY')
if not openai_key:
    raise ValueError('Defina a variável de ambiente OPENAI_API_KEY')


In [28]:
# -------------------------------------------------------------
# Importações para o RAG de código
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# -------------------------------------------------------------
# Defina o caminho do repositório que contém o código a ser indexado
repo_path = '/Users/andreyhitoshi1997/Desktop/Projetos/clinic-recomend-AI'  # ajuste se necessário
# -------------------------------------------------------------
# Cria um DirectoryLoader que usa TextLoader (texto puro) para cada arquivo .dart
loader = DirectoryLoader(
    repo_path,
    glob='**/*.dart',            # padrão de busca (todos os .dart recursivamente)
    recursive=True,
    loader_cls=TextLoader,       # <‑‑ usa TextLoader em vez do default UnstructuredLoader
)
# -------------------------------------------------------------
# Carrega os documentos
documents = loader.load()
print(f'Carregados {len(documents)} arquivos de código do repositório')

Carregados 30 arquivos de código do repositório


In [29]:
# Divide o código em pedaços menores (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)
print(f'Gerados {len(splits)} chunks de código')


Gerados 186 chunks de código


In [30]:
# Cria embeddings com o modelo OpenAI e armazena em Chroma vector store
embeddings = OpenAIEmbeddings(openai_api_key=openai_key)
vectordb = Chroma.from_documents(splits, embeddings)
retriever = vectordb.as_retriever(search_type='similarity', search_kwargs={'k': 4})


In [31]:
# Prompt para a cadeia de QA (focado em código)
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Você é um assistente especializado em analisar código Dart e responder perguntas usando apenas o conteúdo do trecho de código fornecido.'),
    ('human', '{input}')
])


In [ ]:
# Monta a cadeia de QA — LangChain >= 1.x (usa langchain_classic)
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Documento-chain (combina documentos usando o prompt)
doc_chain = create_stuff_documents_chain(
    llm=ChatOpenAI(model='gpt-4o-mini', openai_api_key=openai_key),
    prompt=prompt
)

# Retrieval-chain (recupera documentos relevantes e os passa ao doc_chain)
qa = create_retrieval_chain(
    retriever=retriever,
    combine_documents_chain=doc_chain
)


In [ ]:
# Exemplo de consulta ao código do repositório
query = 'Qual a estrutura da classe ClinicListScreen?'
result = qa.invoke({'input': query})
print(result['answer'])
